# AETHER STT — phase 1 (CTC branch) training notebook

Clones `aether-v3` from GitHub and runs the CTC-only pipeline: frozen Mimi encoder → semantic codes → `AetherSpeech` transformer → CTC head, on LibriSpeech.

**Run top to bottom. Do not skip Section 1 (smoke test)** — it is the first real end-to-end run of this pipeline; it has only been statically reviewed, not executed, before this notebook.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/karl4th/aether-v3.git"
REPO_DIR = "aether-v3"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

In [ ]:
import importlib.util
import subprocess
import sys


def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)


# Most GPU notebook images already ship a CUDA-matched torch build — don't
# clobber it. Only install if genuinely missing.
if importlib.util.find_spec("torch") is None:
    pip_install("torch")
if importlib.util.find_spec("torchaudio") is None:
    pip_install("torchaudio")

pip_install(
    "transformers>=4.53",
    "datasets>=2.19",
    "soundfile",
    "jiwer",
    "pyyaml",
    "numpy",
    "tqdm",
)

In [ ]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "src"))

import torch

from aether_v3.config import load_config

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
print("using device:", DEVICE)

## 1. Smoke test (`hf-internal-testing/librispeech_asr_dummy`)

A handful of utterances only — this is **not** meant to produce a usable model. It exists to prove the full pipeline actually runs without shape/dtype errors: resample → Mimi encode → byte-encode → cache → train → decode.

If this cell (or the next) errors out, stop and fix it before touching the real LibriSpeech run — the same code path is reused there, just with much more data and a much longer feedback loop.

In [ ]:
from aether_v3.data.mimi_cache import prepare_cache

dummy_config = load_config("configs/ctc_dummy.yaml")
prepare_cache(dummy_config, device=DEVICE)

In [ ]:
from aether_v3.training.train_ctc import run_training

run_training(dummy_config)

### Sanity-check the smoke test

With only a handful of examples, don't expect a low WER — just check: loss trends down, and decoded hypotheses look like *something* related to English text (not literal garbage bytes) by the last eval.

In [ ]:
import json

with open(f"{dummy_config.train.output_dir}/log.jsonl") as f:
    rows = [json.loads(line) for line in f]

for row in rows[-10:]:
    print(row)

## 2. Real run: LibriSpeech clean-100 + clean-360

Only proceed once Section 1 looks sane. This downloads + caches the full LibriSpeech train/dev/test splits (several GB) then trains for real. Meant to run unattended on a rented GPU for a long time — check `configs/ctc_base.yaml` and adjust `train.max_steps` / `train.batch_size` for your hardware before launching.

In [ ]:
real_config = load_config("configs/ctc_base.yaml")
prepare_cache(real_config, device=DEVICE)

### Train

**Single GPU:** run the Python cell below.

**Multiple GPUs on this machine:** don't use the Python cell — use the shell cell instead (`torchrun` spawns its own processes; the training loop auto-detects its environment variables and switches to DDP with no code changes).

In [ ]:
run_training(real_config)

In [ ]:
# Multi-GPU alternative to the cell above — edit nproc_per_node, then run this
# cell instead of the plain `run_training(real_config)` call.
# NPROC = 4
# !torchrun --nproc_per_node={NPROC} -m aether_v3.training.train_ctc --config configs/ctc_base.yaml

## 3. Monitor training

Re-run this cell any time (even from a second notebook while the cell above is still training) to see the latest loss/WER/CER curves from `log.jsonl`.

In [ ]:
import json

import matplotlib.pyplot as plt

with open(f"{real_config.train.output_dir}/log.jsonl") as f:
    rows = [json.loads(line) for line in f]

train_rows = [r for r in rows if "loss" in r and "eval_loss" not in r]
eval_rows = [r for r in rows if "eval_cer" in r]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot([r["step"] for r in train_rows], [r["loss"] for r in train_rows])
axes[0].set_title("train loss")
axes[0].set_xlabel("step")

axes[1].plot([r["step"] for r in eval_rows], [r["eval_cer"] for r in eval_rows], label="CER")
axes[1].plot([r["step"] for r in eval_rows], [r["eval_wer"] for r in eval_rows], label="WER")
axes[1].set_title("dev CER / WER")
axes[1].set_xlabel("step")
axes[1].legend()
plt.show()

if eval_rows:
    best = min(eval_rows, key=lambda r: r["eval_cer"])
    print("best eval so far:", best)